In [1]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import re
import torch

tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("./checkpoint-17000")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer.src_lang = "ne_XX"
tokenizer.tgt_lang = "en_XX"

c:\NLTTA\NLTTA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def pre_process(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()  
        
        return text

In [ ]:
import re

def remove_word_prefixes(text, prefixes):
    pattern = r'^(' + '|'.join(re.escape(p) for p in prefixes) + r')\b\s*'
    return re.sub(pattern, '', text)

In [4]:
def translate(text):
    inputs = tokenizer(pre_process(text), return_tensors="pt")
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
        max_length=128,
        num_beams=5
    )

    return remove_word_prefixes(str(tokenizer.decode(generated[0], skip_special_tokens=True)), ['e', 'ne'])
                

In [12]:
translate("samsad ghar")

'parliament house'

In [6]:
translate("mantri sabai le sammelan ma vag liye")

'minister all took part in the conference'

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from bert_score import score


In [14]:
import pandas as pd

In [20]:
df = pd.read_excel("english-nepali-roman.xlsx")
df

,Unnamed: 0,english_sent,nepali_sent,roman_sent
0,0,"It happened after the death of Saul, when Davi...",दाऊदले अमालेकीहरूलाई हराएर पछि सिकलग गए। यो शा...,dAUdale amAlekIharUlAI harAera paChi sikalaga ...
1,1,"it happened on the third day, that behold, a m...",तब तेस्रो दिनमा एउटा जवान सैनिक सिकलगमा आयो। त...,taba tesro dinamA euTA javAna sainika sikalaga...
2,2,"David said to him, ""Where do you come from?"" H...","दाऊदले त्यसलाई सोधे, “तिमी कहाँबाट आयौ?” त्यस ...","dAUdale tyasalAI sodhe, “timI kahA.NbATa Ayau?..."
3,3,"David said to him, ""How did it go? Please tell...","दाऊदले भने, “मलाई भन, के भयो?” त्यसले भन्यो, “...","dAUdale bhane, “malAI bhana, ke bhayo?” tyasal..."
4,4,"David said to the young man who told him, ""How...","दाऊदले त्यस सैनिकलाई भने, “तिमीले कसरी जान्यौ ...","dAUdale tyasa sainikalAI bhane, “timIle kasarI..."
...,...,...,...,...
151932,151932,(b) \tReporting will be regularised by develop...,(ख) आयोजनाहरुको प्रकृति अनुरुप अनुगमन तथा मूल्...,(kha) AyojanAharuko prakRRiti anurupa anugaman...
151933,151933,"(c) \tImpartial on-going, mid -term and impact...",(ग) उपयुक्तताअनुरुप आयोजनाहरुको निष्पक्ष अद्या...,(ga) upayuktatAanurupa AyojanAharuko niShpakSh...
151934,151934,(d) \tThe monitoring and evaluation data will ...,(घ) क्रमिक रुपमा अनुगमन तथा मूल्याङ्कन तथ्याङ्...,(gha) kramika rupamA anugamana tathA mUlyA~Nka...
151935,151935,(e) \tAppropriate rules and regulations will b...,्(ँङ) केन्द्रीय अनुगमन तथा मूल्याङ्कन व्यवस्था...,(.N~Na) kendrIya anugamana tathA mUlyA~Nkana v...


In [23]:
def trim_to_1024_chars(sentence):
    return sentence[:1024] if len(sentence) > 1024 else sentence

In [24]:
roman_nepali_excel = [trim_to_1024_chars(text) for text in df['roman_sent'][150000:155000]]
english_nepali_excel = [trim_to_1024_chars(text) for text in df['english_sent'][150000:155000]]

In [26]:
hypotheses_excel = [
]

for text in roman_nepali_excel:
    translated_text = translate(text)
    hypotheses_excel.append(translated_text)

tokenized_references_excel = [[ref.split()] for ref in english_nepali_excel]  
tokenized_hypotheses_excel = [hyp.split() for hyp in hypotheses_excel]    

smoothing_fn = SmoothingFunction().method1

#corpus-level BLEU
bleu_score = corpus_bleu(tokenized_references_excel, tokenized_hypotheses_excel, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")

#sentence-level BLEU scores
print("\nSentence-level BLEU scores:")
for i, (ref, hyp) in enumerate(zip(tokenized_references_excel, tokenized_hypotheses_excel)):
    sentence_score = sentence_bleu(ref, hyp, smoothing_function=smoothing_fn)
    print(f"Sentence {i+1}: {sentence_score * 100:.2f}")

KeyboardInterrupt: 

In [ ]:
roman_nepali_sentences = [
    "vidharthi ko aandolan",
    "yo kitab ramro cha",
    "ma school janchu",
    "samsad ko baithaak",
    "hami khelna janchau",
    "maile khana khaye",
    "yo ghar dherai thulo cha",
    "tyo manche mero saathi ho",
    "uniharu kheliraheka chan",
    "ma nepali ho",
    "timro ghar ramro cha",
    "unko naam anil ho",
    "hami school ganchau",
    "tyo film malaai dherai manparcha",
    "saarkaar le naya niti ghoshana gareko cha",
    "rajnitik paristhiti badalne cha",
    "bhukampa le dherai chaati gareko cha",
    "chimekhi desh sanga sambandha jhan ramro bhayo",
    "sainik haaru prashikshan ma chan",
    "nayaa kanun samsad bata pass bhayeko cha",
    "udyog haaru le aarthiik pragati gareka chan",
    "mantri sabai le sammelan ma vag liye"
]

references = [
    "students movement",
    "this book is good",
    "i go to school",
    "meeting of parliament",
    "we go to play",
    "i ate food",
    "this house is very big",
    "that person is my friend",
    "they are playing",
    "i am Nepali",
    "your house is nice",
    "his name is Anil",
    "we go to school",
    "i like that movie very much",
    "the government has announced a new policy",
    "the political situation is going to change",
    "the earthquake caused major damage",
    "relations with neighboring countries have strengthened",
    "soldiers are in training",
    "the new law was approved by the parliament",
    "industries have made economic progress",
    "the ministers participated in the conference"
]

hypotheses = [
]

for text in roman_nepali_sentences:
    translated_text = translate(text)
    hypotheses.append(translated_text)

tokenized_references = [[ref.split()] for ref in references]  
tokenized_hypotheses = [hyp.split() for hyp in hypotheses]    

smoothing_fn = SmoothingFunction().method1

#corpus-level BLEU
bleu_score = corpus_bleu(tokenized_references, tokenized_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")

#sentence-level BLEU scores
print("\nSentence-level BLEU scores:")
for i, (ref, hyp) in enumerate(zip(tokenized_references, tokenized_hypotheses)):
    sentence_score = sentence_bleu(ref, hyp, smoothing_function=smoothing_fn)
    print(f"Sentence {i+1}: {sentence_score * 100:.2f}")


Corpus BLEU Score: 55.78

Sentence-level BLEU scores:
Sentence 1: 31.62
Sentence 2: 100.00
Sentence 3: 100.00
Sentence 4: 56.23
Sentence 5: 100.00
Sentence 6: 56.23
Sentence 7: 66.87
Sentence 8: 28.57
Sentence 9: 56.23
Sentence 10: 24.03
Sentence 11: 39.76
Sentence 12: 39.76
Sentence 13: 100.00
Sentence 14: 100.00
Sentence 15: 100.00
Sentence 16: 17.10
Sentence 17: 7.73
Sentence 18: 54.75
Sentence 19: 100.00
Sentence 20: 6.85
Sentence 21: 26.59
Sentence 22: 16.35


In [187]:
hypotheses


['students movement',
 'this book is good',
 'i go to school',
 'meeting of parliament',
 'we go to play',
 'i ate food',
 'this house is very large',
 'that man is my friend',
 'they are playing',
 'i am nepali',
 'your house is good',
 'his name is anil',
 'we go to school',
 'i like that movie very much',
 'the government has announced a new policy',
 'the political situation will be changed',
 'the earthquake has done too much damage',
 'relations with neighboring countries improved',
 'soldiers are in training',
 'new law has passed from parliament',
 'industry has made economic progress',
 'minister all took part in the conference']

In [ ]:
P, R, F1 = score(hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")
print(f"Average BERTScore P: {P.mean().item():.4f}")
print(f"Average BERTScore R: {R.mean().item():.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:02<00:00,  2.13s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 42.69it/s]

done in 2.17 seconds, 10.12 sentences/sec
Average BERTScore F1: 0.9813
Average BERTScore P: 0.9813
Average BERTScore R: 0.9813


**Comparison with Gemini output** Gemini 2.5 Flash (since it is the most used model for new and free users)

In [189]:
gemini_hypotheses = [
    "Student protest/movement",
    "This book is good",
    "I go to school",
    "Parliament meeting",
    "We go to play",
    "I ate food",
    "This house is very big",
    "That person is my friend",
    "They are playing",
    "I am Nepali",
    "Your house is good",
    "His/Her name is Anil",
    "We go to school",
    "I like that film very much",
    "The government has announced a new policy",
    "The political situation will change",
    "The earthquake has caused a lot of damage",
    "Relations with neighboring countries have improved further",
    "Soldiers are in training",
    "A new law has been passed by the parliament",
    "Industries have made economic progress",
    "All ministers participated in the conference"
]

tokenized_gemini_hypotheses = [hyp.lower().split() for hyp in gemini_hypotheses]    

bleu_score = corpus_bleu(tokenized_references, tokenized_gemini_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")


P, R, F1 = score(gemini_hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")


Corpus BLEU Score: 61.36


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.55it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<?, ?it/s]

done in 0.65 seconds, 33.78 sentences/sec
Average BERTScore F1: 0.9790


**Deepseek R1** DeepSeek-V3

In [190]:
deepseek_hypotheses = [
    "The student's protest/movement",
    "This book is good",
    "I go to school",
    "Parliament's session/meeting",
    "We are going to play",
    "I ate food",
    "This house is very big",
    "That person is my friend",
    "They are playing",
    "I am Nepali",
    "Your house is nice",
    "His/Her name is Anil",
    "We go to school",
    "I really like that movie",
    "The government has announced a new policy",
    "The political situation is changing",
    "The earthquake caused massive destruction",
    "Relations with neighboring countries have improved further",
    "The soldiers are in training",
    "The new law has been passed by the parliament",
    "Industries have made economic progress",
    "All ministers participated in the conference"
]

tokenized_deepseek_hypotheses = [hyp.lower().split() for hyp in deepseek_hypotheses]    

bleu_score = corpus_bleu(tokenized_references, tokenized_deepseek_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")


P, R, F1 = score(deepseek_hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")


Corpus BLEU Score: 64.29


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 252.52it/s]

done in 0.65 seconds, 33.92 sentences/sec
Average BERTScore F1: 0.9713


**ChatGpt** GPT-4o

In [191]:
chatgpt_hypotheses = [
    "Students' movement",
    "This book is good",
    "I go to school",
    "Parliament meeting",
    "We go to play",
    "I ate food",
    "This house is very big",
    "That person is my friend",
    "They are playing",
    "I am Nepali",
    "Your house is nice",
    "His name is Anil",
    "We go to school",
    "I like that movie very much",
    "The government has announced a new policy",
    "The political situation will change",
    "The earthquake has caused a lot of damage",
    "Relations with neighboring countries improved",
    "Soldiers are in training",
    "A new law has been passed by parliament",
    "Industries have made economic progress",
    "All ministers participated in the conference"
]

tokenized_chatgpt_hypotheses = [hyp.lower().split() for hyp in chatgpt_hypotheses]    

bleu_score = corpus_bleu(tokenized_references, tokenized_chatgpt_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")


P, R, F1 = score(chatgpt_hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")


Corpus BLEU Score: 68.02


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<?, ?it/s]

done in 0.61 seconds, 36.05 sentences/sec
Average BERTScore F1: 0.9832


In [7]:
references = [
    "students movement",
    "this book is good",
    "i go to school",
    "meeting of parliament",
    "we go to play",
    "i ate food",
    "this house is very big",
    "that person is my friend",
    "they are playing",
    "i am Nepali",
    "your house is nice",
    "his name is Anil",
    "we go to school",
    "i like that movie very much",
    "the government has announced a new policy",
    "the political situation is going to change",
    "the earthquake caused major damage",
    "relations with neighboring countries have strengthened",
    "soldiers are in training",
    "the new law was approved by the parliament",
    "industries have made economic progress",
    "the ministers participated in the conference"
]
roman_nepali_sentences = [
    "vidharthi ko aandolan",
    "yo kitab ramro cha",
    "ma school janchu",
    "samsad ko baithaak",
    "hami khelna janchau",
    "maile khana khaye",
    "yo ghar dherai thulo cha",
    "tyo manche mero saathi ho",
    "uniharu kheliraheka chan",
    "ma nepali ho",
    "timro ghar ramro cha",
    "unko naam anil ho",
    "hami school ganchau",
    "tyo film malaai dherai manparcha",
    "saarkaar le naya niti ghoshana gareko cha",
    "rajnitik paristhiti badalne cha",
    "bhukampa le dherai chaati gareko cha",
    "chimekhi desh sanga sambandha jhan ramro bhayo",
    "sainik haaru prashikshan ma chan",
    "nayaa kanun samsad bata pass bhayeko cha",
    "udyog haaru le aarthiik pragati gareka chan",
    "mantri sabai le sammelan ma vag liye"
]
tokenized_references = [[ref.split()] for ref in references]  
smoothing_fn = SmoothingFunction().method1

print(len(roman_nepali_sentences), len(references))

22 22


**LLAMA 2**(Open source Model)

In [10]:
llama_hypotheses = [
    "A movement for students",
   "How many books do you have?",
   "I am going to school",
   "Meeting at the association",
   "We play together",
   "Eating at home",
   "My house is far away",
   "You are my dear friend",
   "Playing together happily",
   "I am Nepali",
   "Your house is beautiful",
   "His name is Anil",
   "I went to school",
   "Going to see a movie tomorrow",
   "New government has brought good policies",
   "Political situation changes rapidly",
   "Collecting money for education",
   "Understanding the relationship between countries and nations",
   "Military training for children",
   "New law passed by the assembly",
   "Industrialization and economic development",
   "Ministers gathered for a meeting"
]


tokenized_llama_hypotheses = [hyp.lower().split() for hyp in llama_hypotheses]    

bleu_score = corpus_bleu(tokenized_references, tokenized_llama_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")


P, R, F1 = score(llama_hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")


Corpus BLEU Score: 4.75


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:02<00:00,  2.16s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 37.08it/s]

done in 2.20 seconds, 10.01 sentences/sec
Average BERTScore F1: 0.9110


**LLAM 4**(Maverick: 400Billion Parameters, Meta Largest capable AI)

In [11]:
llama4_hypotheses = [
    "Student's movement",
    "This book is good",
    "I go to school",
    "Parliament meeting",
    "We go to play",
    "I have eaten food",
    "This house is very big",
    "That man is my friend",
    "They are playing",
    "I am Nepali",
    "Your house is good",
    "His name is Anil",
    "We go to school",
    "I like that movie a lot",
    "The government has announced a new policy",
    "The political situation is changing",
    "The earthquake caused a lot of damage",
    "The relationship with neighboring countries has improved",
    "Soldiers are in training",
    "The new law has been passed by parliament",
    "The industries have made economic progress",
    "All ministers spoke at the conference"
]


tokenized_llama4_hypotheses = [hyp.lower().split() for hyp in llama4_hypotheses]    

bleu_score = corpus_bleu(tokenized_references, tokenized_llama4_hypotheses, smoothing_function=smoothing_fn)
print(f"Corpus BLEU Score: {bleu_score * 100:.2f}")


P, R, F1 = score(llama4_hypotheses, references, lang="en", verbose=True)
print(f"Average BERTScore F1: {F1.mean().item():.4f}")


Corpus BLEU Score: 53.86


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 500.69it/s]

done in 0.61 seconds, 36.18 sentences/sec
Average BERTScore F1: 0.9754
